# 1. Coxeter $(E_6,P_6)$-matroids and their classification

This notebook follows Section 2 of the extended abstract *Coxeter matroids of type $(E_6,P_6)$ and the $E_6$-Grassmannian*. It covers

1. the Gosset polytope $2_{21}$ and the $27$ lines on a cubic surface,
2. the characterization of $(E_6,P_6)$-matroids by tritangent planes (Theorem 2.4),
3. the classification: $1\,335\,373$ matroids in $230$ orbits under $W(E_6)$ (Theorem 2.5),
4. the three matroids $Q_1,Q_2,Q_3$ that are complements of triads.

All functions are defined in `src/gosset.jl` and `src/matroids.jl`. Loading the code takes a moment, because it lists the $51\,840$ elements of $W(E_6)$.

In [6]:
#using Pkg
#Pkg.instantiate()
include("../src/E6P6CoxeterMatroids.jl");

## The Gosset polytope and the 27 lines

The $27$ vertices of $2_{21}$ are named after the $27$ lines on a smooth cubic surface, and are always listed in the order

$$a_1,\dots,a_6,\quad b_1,\dots,b_6,\quad c_{12},c_{13},\dots,c_{56}.$$

A set of lines is stored as a sorted vector of indices in `1:27`. For computations we use the coordinates

$$
a_i = 2e_i + 2e_7, \quad b_i = 2e_i + 2e_8, \quad c_{ij} = \Bigl(\sum_{k=1}^{8} e_k\Bigr) - 2e_i - 2e_j
$$

in $\mathbb{R}^8$, which are the rows of the matrix `Gosset_lines`. This is an affine image of the set $\mathcal{L}\subseteq M$ of the abstract.

In [5]:
println(line_labels)
P221 = convex_hull(Gosset_lines)
@show dim(P221)
@show f_vector(P221);

["a1", "a2", "a3", "a4", "a5", "a6", "b1", "b2", "b3", "b4", "b5", "b6", "c12", "c13", "c14", "c15", "c16", "c23", "c24", "c25", "c26", "c34", "c35", "c36", "c45", "c46", "c56"]
dim(P221) = 6
f_vector(P221) = ZZRingElem[27, 216, 720, 1080, 648, 99]


The inner product on $M$ separates pairs of lines into *skew* pairs, $\langle x,y\rangle = \tfrac13$, and *meeting* pairs, $\langle x,y\rangle=-\tfrac23$. In the coordinates above, two lines meet exactly when their dot product is $0$; this is recorded in the matrix `meets`. For a line $y$,

* `neighbours(y)` is the set $N(y)$ of the $16$ lines skew to $y$: the neighbours of $y$ in $2_{21}$,
* `lozenge(y)` is the set $\lozenge(y)$ of the $10$ lines meeting $y$: the vertices of the cross-polytope facet opposite to $y$,
* `tritangent_pairs[y]` lists the $5$ antipodal pairs $\{x,x^*\}$ of $\lozenge(y)$, that is, the tritangent planes $\{x,x^*,y\}$ through $y$.

Here is Example 2.2, for $y=a_6$.

In [7]:
y = 6
@show line_labels[y]
@show line_labels[lozenge(y)]
@show line_labels[neighbours(y)]
[line_labels[p] for p in tritangent_pairs[y]]

line_labels[y] = "a6"
line_labels[lozenge(y)] = ["b1", "b2", "b3", "b4", "b5", "c16", "c26", "c36", "c46", "c56"]
line_labels[neighbours(y)] = ["a1", "a2", "a3", "a4", "a5", "b6", "c12", "c13", "c14", "c15", "c23", "c24", "c25", "c34", "c35", "c45"]


5-element Vector{Vector{String}}:
 ["b1", "c16"]
 ["b2", "c26"]
 ["b3", "c36"]
 ["b4", "c46"]
 ["b5", "c56"]

In [8]:
# 216 edges of 2_21 = skew pairs; 45 tritangent planes
@show count(!meets[i, j] for i in 1:27 for j in i+1:27)
@show length(tritangent_planes);

count((!(meets[i, j]) for i = 1:27 for j = i + 1:27)) = 216
length(tritangent_planes) = 45


The Weyl group $W(E_6)$ permutes the $27$ lines. `W_perms` lists its elements as permutations of `1:27`. They preserve the incidence relation.

In [9]:
@show length(W_perms)
@show all(meets[g, g] == meets for g in W_perms);

length(W_perms) = 51840
all((meets[g, g] == meets for g = W_perms)) = true


## Three descriptions of matroids (Theorem 2.4)

A nonempty set $Q$ of lines is an $(E_6,P_6)$-matroid if every edge of $\operatorname{conv}(Q)$ is parallel to a root. Theorem 2.4 gives equivalent conditions. Three tests are implemented:

* `is_matroidal(Q)`: the definition, by computing the edges of $\operatorname{conv}(Q)$;
* `is_E6P6_matroid(Q)`: condition (iii), no $\lozenge(y)$ contains exactly one antipodal pair inside $Q$;
* `has_tritangent_exchange(Q)`: condition (iv), the tritangent exchange property.

In [10]:
Q = Q1                                   # the matroid Q_1 of the abstract
println(line_labels[Q])
@show is_matroidal(Q), is_E6P6_matroid(Q), has_tritangent_exchange(Q)

Q = line_indices("a1", "b2")             # two meeting lines: not a matroid
@show is_matroidal(Q), is_E6P6_matroid(Q), has_tritangent_exchange(Q);

["a1", "a2", "a3", "a4", "a5", "a6", "b1", "b2", "b3", "b4", "b5", "b6", "c12", "c13", "c23", "c45", "c46", "c56"]
(is_matroidal(Q), is_E6P6_matroid(Q), has_tritangent_exchange(Q)) = (true, true, true)
(is_matroidal(Q), is_E6P6_matroid(Q), has_tritangent_exchange(Q)) = (false, false, false)


We compare the three tests on representatives of the $W(E_6)$-orbits of subsets of $\mathcal{L}$ (computed below). The polytope test is slow, so by default it runs on every 20th representative; set `step = 1` to run it on all of them.

In [11]:
subs = read_sets(joinpath(datadir, "subset_orbit_reps.dat"))
@show length(subs)
@show all(is_E6P6_matroid(Q) == has_tritangent_exchange(Q) for Q in subs)

step = 20
@show all(is_matroidal(Q) == is_E6P6_matroid(Q) for Q in subs[1:step:end]);

length(subs) = 5484
all((is_E6P6_matroid(Q) == has_tritangent_exchange(Q) for Q = subs)) = true
all((is_matroidal(Q) == is_E6P6_matroid(Q) for Q = subs[1:step:end])) = true


## The classification (Theorem 2.5)

The orbits of subsets are computed in `scripts/classify.jl`. Each orbit is represented by its lexicographically smallest element. Every orbit of $(k+1)$-subsets contains a set $R\cup\{v\}$ with $R$ a representative of size $k$, so representatives are grown one line at a time, up to size $13$; larger sets are complements of smaller ones. This is the longest computation of the project (hours), so the result is stored in `data/`:

* `data/subset_orbit_reps.dat`: the $5\,484$ orbits of subsets of sizes $1,\dots,26$ (with $\emptyset$ and $\mathcal{L}$ there are $5\,486$),
* `data/matroid_orbit_reps.dat`: the $230$ orbits of matroids, loaded as `E6_mats`.

As a check of the first file, the orbit sizes must add up to $2^{27}$ (about two minutes).

In [12]:
@time 2 + sum(orbit_size, subs) == 2^27

 38.069883 seconds (570.32 M allocations: 44.512 GiB, 10.73% gc time, 0.14% compilation time)


true

Applying Theorem 2.4(iii) to the representatives gives the matroids: $229$ proper ones, and $\mathcal{L}$ itself.

In [13]:
@show filter(is_E6P6_matroid, subs) == E6_mats[1:229]
@show E6_mats[230] == collect(1:27)
@show length(E6_mats)
@show sum(orbit_size, E6_mats);

filter(is_E6P6_matroid, subs) == E6_mats[1:229] = true
E6_mats[230] == collect(1:27) = true
length(E6_mats) = 230
sum(orbit_size, E6_mats) = 1335373


The table of Theorem 2.5: the number of orbits by the dimension of $\operatorname{conv}(Q)$.

In [14]:
dims = [dim(convex_hull(Gosset_lines[Q, :])) for Q in E6_mats]
[d => count(==(d), dims) for d in 0:6]

7-element Vector{Pair{Int64, Int64}}:
 0 => 1
 1 => 1
 2 => 2
 3 => 4
 4 => 11
 5 => 30
 6 => 181

## The graph $\Gamma_Q$ and triads

$\Gamma_Q$ is the graph on $Q$ joining meeting lines; `valency(Q, q)` is the degree of $q$. Exactly three orbits are $7$-regular. They have $18$ elements, and their complements are *triads*: unions of three pairwise disjoint tritangent planes.

In [15]:
regular = [Q for Q in E6_mats if all(valency(Q, q) == 7 for q in Q)]
@show length.(regular)

triads = [setdiff(1:27, Q) for Q in regular]
is_triad(T) = any(sort(vcat(p, q, r)) == T for p in tritangent_planes, q in tritangent_planes, r in tritangent_planes)
@show is_triad.(triads)
@show orbit_size.(triads);              # 120 + 2160 + 2880 = 5160 triads

length.(regular) = [18, 18, 18]
is_triad.(triads) = Bool[1, 1, 1]
orbit_size.(triads) = [120, 2160, 2880]


$Q_1$ is the complement of a triad in the orbit of size $120$. It plays a special role in the next notebook.

In [16]:
@show minimal_rep(Q1) == regular[1]
G = E6P6_automorphism_group(Q1)         # the stabilizer of Q_1 in W(E6)
@show order(G)
describe(G)

minimal_rep(Q1) == regular[1] = true
order(G) = 432


"((S3 x S3) : C2) x S3"